# 任务7：生成模块与提示词工程第一部分

In [1]:
import pandas as pd
from pathlib import Path
import pickle
import json


from medical_rag.reranker import MedicalBGECrossEncoderReranker
from medical_rag.context_assembler import ContextAssembler
from medical_rag.generation_pipeline import MedicalGenerationPipeline
from medical_rag.llm_generator import LLMGenerator
from medical_rag.prompts import (
    MEDICAL_PROMPT_STAGES,
    render_prompt
)

from medical_rag.generation_utils import (
    display_generation_result,
    run_medical_generation_test
)

e:\anaconda3\envs\medrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 交互验证

In [2]:
demo_results = pd.DataFrame([
    {"text": "二甲双胍可能改善部分患者的代谢指标，但研究人群和随访时间有限。",
     "final_score": 0.95, "source_title": "Study A",
     "journal": "Nature Communications", "publication_year": 2024,
     "pmid": "10001", "doc_id": "doc_a", "chunk_index": 0,
     "vector_id": "doc_a_chunk_0000"},
    {"text": "二甲双胍可能改善部分患者的代谢指标，但研究人群和随访时间有限。",
     "final_score": 0.82, "source_title": "Study A",
     "vector_id": "doc_a_chunk_0001"},
    {"text": "另一项研究评估了不同人群中的心血管结局，结果仍存在不确定性。",
     "final_score": 0.91, "source_title": "Study B",
     "journal": "The Lancet", "publication_year": 2023,
     "pmid": "10002", "vector_id": "doc_b_chunk_0000"},
])

context_tokenizer = (
    MedicalBGECrossEncoderReranker
    .load_tokenizer(
        model_name="BAAI/bge-reranker-base"
    )
)


assembler = ContextAssembler(
    tokenizer = context_tokenizer,
    max_context_tokens=500,
    dedup_threshold=0.85,
    max_chunks_per_source=2,
    diversity_penalty=0.15,
)
assembled_demo = assembler.assemble_context(demo_results)
display(pd.DataFrame([assembled_demo["metadata"]]))
print(assembled_demo["context_text"])

,total_chunks_retrieved,unique_chunks_after_dedup,chunks_selected,estimated_tokens,chunk_sources
0,3,2,2,81,"{'Study A': 1, 'Study B': 1}"


[证据 1 | 来源: Study A | 相关性: 0.9500]
二甲双胍可能改善部分患者的代谢指标，但研究人群和随访时间有限。

[证据 2 | 来源: Study B | 相关性: 0.9100]
另一项研究评估了不同人群中的心血管结局，结果仍存在不确定性。


## 渲染第一阶段提示词并查看

In [3]:
question = "二甲双胍对心血管结局有何影响？"
evidence_request = render_prompt(
    "evidence_evaluator",
    question=question,
    context=assembled_demo["context_text"],
)
print("temperature:", evidence_request["temperature"])
print("max_tokens:", evidence_request["max_tokens"])
print("\n--- System Prompt ---\n", evidence_request["messages"][0]["content"])
print("\n--- User Prompt ---\n", evidence_request["messages"][1]["content"])

temperature: 0.1
max_tokens: 1800

--- System Prompt ---
 你是循证医学证据评估专家。只能依据给定上下文，不得补造论文、数据或结论。区分研究设计、人群、干预/暴露、对照、结局和不确定性；识别冲突、偏倚与适用性。信息缺失时明确写‘证据不足’，不直接向患者下诊断。

--- User Prompt ---
 用户问题：
二甲双胍对心血管结局有何影响？

检索上下文：
[证据 1 | 来源: Study A | 相关性: 0.9500]
二甲双胍可能改善部分患者的代谢指标，但研究人群和随访时间有限。

[证据 2 | 来源: Study B | 相关性: 0.9100]
另一项研究评估了不同人群中的心血管结局，结果仍存在不确定性。

请输出：1.相关证据及编号；2.研究类型、人群和核心结果；3.证据质量及理由；4.冲突和局限；5.可支持与不能支持的结论。事实结论必须标注[证据 N]。


# 任务8：生成模块与提示词工程第二部分

# LLM Generation流水线

## 各部分模块初始化

In [3]:
# 初始化ollama

OLLAMA_MODEL_NAME = "deepseek-r1-7b"

llm_generator = LLMGenerator(
    model_name=OLLAMA_MODEL_NAME,
    base_url="http://localhost:11434",
    timeout=600,
    default_temperature=0.2,
    default_max_tokens=1600,
    keep_alive="10m",
    test_connection=True,
)

In [4]:
context_assembler = ContextAssembler(
    tokenizer=context_tokenizer,
    tokenizer_name=None,
    max_context_tokens=6000,
    dedup_threshold=0.85,
    max_chunks_per_source=2,
    diversity_penalty=0.15,
)


generation_pipeline = (
    MedicalGenerationPipeline(
        context_assembler=(
            context_assembler
        ),
        prompt_stages=(
            MEDICAL_PROMPT_STAGES
        ),
        llm_generator=(
            llm_generator
        ),
    )
)

print("MedicalGenerationPipeline初始化完成")

MedicalGenerationPipeline初始化完成


## 单词LLM生成测试

In [7]:
# 单次生成接口测试

connection_test = (
    llm_generator.test_connection()
)

display(
    pd.DataFrame([
        connection_test
    ])
)


simple_generation_test = (
    llm_generator.generate(
        prompt=(
            "请返回一个JSON对象，包含status和message字段。"
        ),
        system_prompt=(
            "你是本地模型连接测试助手。"
        ),
        temperature=0.0,
        max_tokens=1024,
        require_json=True,
        json_schema={
            "type": "object",
            "properties": {
                "status": {
                    "type": "string",
                },
                "message": {
                    "type": "string",
                },
            },
            "required": [
                "status",
                "message",
            ],
        },
        think=False,
    )
)

print(
    json.dumps(
        simple_generation_test[
            "parsed_json"
        ],
        ensure_ascii=False,
        indent=2,
    )
)

,connected,model_available,model_name,available_models,latency_seconds
0,True,True,deepseek-r1-7b,[deepseek-r1-7b:latest],0.0088


{
  "model": "deepseek-r1-7b",
  "created_at": "2026-08-19T12:58:50.3079133Z",
  "message": {
    "role": "assistant",
    "content": "{\n  \"message\": \"\",\n  \"status\": \"success\"\n}",
    "thinking": "\n好，我现在需要处理用户的请求。用户说他们是本地模型连接测试助手，要求返回一个JSON对象，包含status和message字段。而且必须符合特定的JSON Schema，不能添加其他内容。\n\n首先，我要理解用户的需求。他们可能是在测试本地模型连接时，需要一个助手来提供反馈。所以，当连接成功时，应该返回一个成功状态；如果失败，返回错误信息。\n\n接下来，我要检查用户提供的JSON Schema是否正确。用户给出的结构是正确的，包含type、properties和required字段。所以，我需要确保返回的JSON符合这个结构。\n\n然后，我需要决定返回什么状态和信息。通常，测试连接成功时，status应该是\"success\"，message是空字符串。如果连接失败，比如网络问题或模型文件不存在，status是\"error\"，message说明原因。\n\n假设用户现在测试连接成功，那么我应该返回status为\"success\"，message为空。这样既符合要求，又简洁明了。\n\n最后，我要确保只返回合法的JSON，不添加任何额外内容，比如解释文字或其他结构。这样，用户就能直接使用这个JSON对象，不需要处理其他部分。\n\n总结一下，我会返回一个包含status为\"success\"，message为空的JSON对象，符合用户提供的Schema。\n"
  },
  "done": true,
  "done_reason": "stop",
  "total_duration": 97793634400,
  "load_duration": 39077197600,
  "prompt_eval_count": 108,
  "prompt_eval_duration": 30287722000,
  "eval_count"

## 完整流程测试

In [5]:
#导入检索结果

input_path = Path(
    r"F:\RAG\data\pipeline_outputs"
    r"\task6_pipeline_output.pkl"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"找不到任务6检索结果：{input_path}"
    )

with input_path.open("rb") as file:
    pipeline_output = pickle.load(file)

print("pipeline_output加载完成")
print("包含字段：")
print(list(pipeline_output.keys()))

pipeline_output加载完成
包含字段：
['original_query', 'query_info', 'rerank_query', 'vector_results', 'bm25_results', 'fused_results', 'final_results', 'statistics']


In [6]:
#将检索结果传给上下文组装器

context_result = (
    context_assembler
    .assemble_context(
        pipeline_output[
            "final_results"
        ]
    )
)

display(
    pd.DataFrame([
        context_result[
            "metadata"
        ]
    ])
)

print(
    context_result[
        "context_text"
    ]
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (528 > 512). Running this sequence through the model will result in indexing errors


,total_chunks_retrieved,unique_chunks_after_dedup,chunks_selected,estimated_tokens,chunk_sources
0,10,10,10,5286,{'A Pilot randomized trial to examine effects ...


[证据 1 | 来源: A Pilot randomized trial to examine effects of a hybrid closed-loop insulin delivery system on neurodevelopmental and cognitive outcomes in adolescents with type 1 diabetes | 相关性: 0.2836]
Title: A Pilot randomized trial to examine effects of a hybrid closed-loop insulin delivery system on neurodevelopmental and cognitive outcomes in adolescents with type 1 diabetes

. Supporting the importance of the frontal cortical results were findings that expected reductions in caudate nucleus volumes were more prominent in the CL group and were correlated with decreases in glucose variability across the entire cohort. The caudate nucleus is a major component of the corpus striatum, and frontal–striatal networks underlie the development of critical cognitive functions that undergo maturation during adolescence and young adulthood30–32. Given prior work from our group showing slower growth of the hippocampus over 18 months is associated with increased exposure to hyperglycemia and with 

In [7]:
# 完整医学流程测试

test_query = (
    "二甲双胍对心血管结局有何影响？"
)

if (
    "pipeline_output" in globals()
    and "final_results"
    in pipeline_output
):
    test_retrieved_docs = (
        pipeline_output[
            "final_results"
        ]
    )

elif "demo_results" in globals():
    test_retrieved_docs = (
        demo_results
    )

else:
    # 没有任务6结果时使用最小演示数据
    test_retrieved_docs = pd.DataFrame([
        {
            "text": (
                "一项研究评估了二甲双胍使用与"
                "心血管结局之间的关系。"
                "研究结果提示可能存在获益，"
                "但不能仅凭观察性关联确认因果关系。"
            ),
            "final_score": 0.93,
            "source_title": (
                "Metformin and "
                "cardiovascular outcomes"
            ),
            "journal": (
                "Nature Communications"
            ),
            "publication_year": 2024,
            "pmid": "10001",
            "doc_id": "demo_doc_1",
            "chunk_index": 0,
            "vector_id": (
                "demo_doc_1_chunk_0000"
            ),
        },
        {
            "text": (
                "另一项研究报告不同研究人群"
                "中的结果并不完全一致，"
                "并指出基线风险和合并用药"
                "可能影响结果。"
            ),
            "final_score": 0.88,
            "source_title": (
                "Heterogeneity of "
                "metformin outcomes"
            ),
            "journal": "The Lancet",
            "publication_year": 2023,
            "pmid": "10002",
            "doc_id": "demo_doc_2",
            "chunk_index": 0,
            "vector_id": (
                "demo_doc_2_chunk_0000"
            ),
        },
    ])


test_result = run_medical_generation_test(
    pipeline=generation_pipeline,
    query=test_query,
    retrieved_docs=test_retrieved_docs,
    evaluate_evidence=True,
    critical_review=True,
)

display_generation_result(test_result)

{
  "model": "deepseek-r1-7b",
  "created_at": "2026-08-19T13:28:34.9484823Z",
  "message": {
    "role": "assistant",
    "content": "{\n  \"conflicts\": [],\n  \"evidence_quality\": \"证据1和证据9是RCT，样本量虽小，但设计合理，数据可靠，支持二甲双胍对心血管结局的负面影响。\",\n  \"limitations\": [\n    \"样本量小，可能无法推广到更大群体\",\n    \"研究对象为青少年，可能在成人中效果不同\"\n  ],\n  \"relevant_evidence_ids\": [1, 9],\n  \"summary\": \"二甲双胍对14-17岁青少年1型糖尿病患者的血糖控制不佳，导致认知功能下降和脑结构变化。\",\n  \"supported_conclusions\": [\n    \"[证据1] 2型双胍对青少年1型糖尿病患者的血糖控制不佳，导致认知功能下降和脑结构变化。\",\n    \"[证据9] 长期使用2型双胍导致青少年1型糖尿病患者的血糖水平持续偏高，影响认知功能和脑结构。\",\n    \"[证据1] 2型双胍通过降低血糖水平来改善心血管结局。\",\n    \"[证据9] 长期使用2型双胍可能导致糖尿病肾病，影响心血管功能。\",\n    \"[证据1] 2型双胍通过降低血糖水平来改善心血管结局，但需谨慎解读样本量的限制。\"\n  ],\n  \"unsupported_conclusions\": []\n}",
    "thinking": "\n好，我现在需要处理用户的问题，他提供了一系列证据，并要求我根据这些证据生成一个合法的JSON输出，内容包括相关证据、研究类型、证据质量、冲突、局限、支持和不支持的结论。同时，事实结论必须标注[证据 N]。\n\n首先，我要仔细阅读用户提供的所有证据，理解每个证据的内容和相关性。用户提供了10个证据，每个都有标题、相关性评分和内容。我的任务是将这些证据分类，并根据它们的重要性来确定哪些是关键的。\n\n证据1是关于2型糖尿病对心血管结局的影响，相关性0.2836。这

,stage,success,elapsed_seconds,prompt_tokens,output_tokens,total_tokens
0,context_assembly,True,0.0669,0,0,5286
1,evidence_evaluator,True,84.1000,5404,1252,6656
2,answer_generator,True,20.8972,1419,454,1873
3,critical_reviewer,True,31.3301,1591,711,2302
4,final_assembler,True,37.5208,1891,820,2711
5,postprocessing,True,0.0006,0,0,0



引用来源


,evidence_id,citation,chunk_id,source,relevance_score,journal,publication_year,pmid,doc_id,chunk_index
0,1,[证据 1],DOC_327ad2b049a6_chunk_0030,A Pilot randomized trial to examine effects of...,0.283625,Nature Communications,2022,,DOC_327ad2b049a6,30
1,2,[证据 2],PMID_35121731_chunk_0076,The gut hormone Allatostatin C/Somatostatin re...,0.282666,Nature Communications,2022,35121731,PMID_35121731,76
2,3,[证据 3],PMID_36030260_chunk_0041,Reversal of the renal hyperglycemic memory in ...,0.281840,Nature Communications,2022,36030260,PMID_36030260,41
3,4,[证据 4],PMID_35551192_chunk_0033,Germline mutations in mitochondrial complex I ...,0.278938,Nature Communications,2022,35551192,PMID_35551192,33
4,5,[证据 5],PMID_35418199_chunk_0078,Sympathetic axonal sprouting induces changes i...,0.277837,Nature Communications,2022,35418199,PMID_35418199,78
5,6,[证据 6],DOC_979625960f1d_chunk_0017,Cutaneous and acral melanoma cross-OMICs revea...,0.277819,Nature Communications,2022,,DOC_979625960f1d,17
6,7,[证据 7],PMID_35121731_chunk_0082,The gut hormone Allatostatin C/Somatostatin re...,0.280644,Nature Communications,2022,35121731,PMID_35121731,82
7,8,[证据 8],PMID_36030260_chunk_0034,Reversal of the renal hyperglycemic memory in ...,0.280501,Nature Communications,2022,36030260,PMID_36030260,34
8,9,[证据 9],DOC_327ad2b049a6_chunk_0002,A Pilot randomized trial to examine effects of...,0.279331,Nature Communications,2022,,DOC_327ad2b049a6,2
9,10,[证据 10],DOC_979625960f1d_chunk_0020,Cutaneous and acral melanoma cross-OMICs revea...,0.277614,Nature Communications,2022,,DOC_979625960f1d,20


查询
二甲双胍对心血管结局有何影响？

最终回答
简明结论：二甲双胍在青少年1型糖尿病患者中的应用可能带来认知功能和脑结构变化的负面影响，但目前证据未能直接支持其对心血管结局的具体影响。建议进一步研究以明确二甲双胍对心血管系统的长期影响。

关键证据：
- [证据 1]：研究显示二甲双胍在青少年1型糖尿病患者中的血糖控制不佳，导致认知功能下降和脑结构变化。尽管样本量小，但设计合理，数据可靠。
- [证据 9]：长期使用二甲双胍可能导致血糖水平持续偏高，影响认知功能和脑结构，甚至可能引发糖尿病肾病，影响心血管功能。

局限：
- 样本量小，可能无法推广到更大群体。
- 研究对象为青少年，可能在成人中效果不同。

实际含义：
二甲双胍在青少年1型糖尿病患者中的应用可能带来认知和心血管功能的负面影响，提示在临床中需谨慎评估其长期效果。

审查意见：
### 逐步解释

1. **问题分析**：用户询问二甲双胍对青少年1型糖尿病患者的心血管结局有何影响。检索到的证据主要集中在认知功能和脑结构变化上，而非直接的心血管影响。

2. **证据评估**：
   - **相关性**：证据1和证据9的相关性较高，显示二甲双胍在青少年1型糖尿病患者中的血糖控制不佳，导致认知功能下降和脑结构变化。
   - **局限性**：样本量小，可能无法推广到更大群体；研究对象为青少年，可能在成人中效果不同。

3. **因果关系**：虽然证据显示二甲双胍导致血糖控制不佳，进而影响认知和脑结构，但没有直接证据连接到心血管结局，因此因果关系可能被夸大。

4. **结论调整**：虽然二甲双胍可能对心血管功能产生负面影响，但目前证据主要集中在认知和脑结构变化上，需要进一步研究来直接评估心血管影响。

5. **建议**：用户应谨慎解读现有证据，并建议进一步的研究来直接评估二甲双胍对心血管结局的影响。

### 最终结论

二甲双胍在青少年1型糖尿病患者中的应用可能带来认知功能和脑结构变化的负面影响，但目前证据未能直接支持其对心血管结局的具体影响。建议进一步研究以明确二甲双胍对心血管系统的长期影响。

## 参考证据

- [证据 1] A Pilot randomized trial to examine effects of a hybrid closed-loop insulin delivery 

,stage,success,elapsed_seconds,prompt_tokens,output_tokens,total_tokens
0,context_assembly,True,0.0669,0,0,5286
1,evidence_evaluator,True,84.1000,5404,1252,6656
2,answer_generator,True,20.8972,1419,454,1873
3,critical_reviewer,True,31.3301,1591,711,2302
4,final_assembler,True,37.5208,1891,820,2711
5,postprocessing,True,0.0006,0,0,0



引用来源


,evidence_id,citation,chunk_id,source,relevance_score,journal,publication_year,pmid,doc_id,chunk_index
0,1,[证据 1],DOC_327ad2b049a6_chunk_0030,A Pilot randomized trial to examine effects of...,0.283625,Nature Communications,2022,,DOC_327ad2b049a6,30
1,2,[证据 2],PMID_35121731_chunk_0076,The gut hormone Allatostatin C/Somatostatin re...,0.282666,Nature Communications,2022,35121731,PMID_35121731,76
2,3,[证据 3],PMID_36030260_chunk_0041,Reversal of the renal hyperglycemic memory in ...,0.281840,Nature Communications,2022,36030260,PMID_36030260,41
3,4,[证据 4],PMID_35551192_chunk_0033,Germline mutations in mitochondrial complex I ...,0.278938,Nature Communications,2022,35551192,PMID_35551192,33
4,5,[证据 5],PMID_35418199_chunk_0078,Sympathetic axonal sprouting induces changes i...,0.277837,Nature Communications,2022,35418199,PMID_35418199,78
5,6,[证据 6],DOC_979625960f1d_chunk_0017,Cutaneous and acral melanoma cross-OMICs revea...,0.277819,Nature Communications,2022,,DOC_979625960f1d,17
6,7,[证据 7],PMID_35121731_chunk_0082,The gut hormone Allatostatin C/Somatostatin re...,0.280644,Nature Communications,2022,35121731,PMID_35121731,82
7,8,[证据 8],PMID_36030260_chunk_0034,Reversal of the renal hyperglycemic memory in ...,0.280501,Nature Communications,2022,36030260,PMID_36030260,34
8,9,[证据 9],DOC_327ad2b049a6_chunk_0002,A Pilot randomized trial to examine effects of...,0.279331,Nature Communications,2022,,DOC_327ad2b049a6,2
9,10,[证据 10],DOC_979625960f1d_chunk_0020,Cutaneous and acral melanoma cross-OMICs revea...,0.277614,Nature Communications,2022,,DOC_979625960f1d,20
